In [ ]:
from sklearn.datasets import make_classification
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

using device: cuda


In [ ]:
# step 1: Create a synthetic classification dataset using sklear
X, y = make_classification(
    n_samples=10,     # number of samples
    n_features=2,     # number of features
    n_informative=2,  # number of informative features
    n_redundant=0,    # number of redundant features
    n_classes=2,      # number of classes
    random_state=42   # random state for reproducibility
)

In [ ]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [ ]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [ ]:
# convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels


  def __len__(self):
    return self.features.shape[0]


  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
dataset = CustomDataset(X, y)

In [ ]:
dataset

In [ ]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
dataloader

In [ ]:
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print("-" * 50)

tensor([[-0.5872, -1.9717],
        [-0.7206, -0.9606]])
tensor([0., 0.])
--------------------------------------------------
tensor([[-1.1402, -0.8388],
        [ 1.7774,  1.5116]])
tensor([0., 1.])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.8997,  0.8344]])
tensor([1., 1.])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-1.9629, -0.9923]])
tensor([1., 0.])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [ 1.0683, -0.9701]])
tensor([0., 1.])
--------------------------------------------------


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)   # .fit_transform is used on the training data
x_test = scaler.transform(x_test)         # .transform() is used on test or validation data.

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


In [ ]:
# Numpy arrays to PyTorch tensors
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)



x_train_tensor = x_train_tensor.to(torch.float32)
x_test_tensor = x_test_tensor.to(torch.float32)
y_train_tensor = y_train_tensor.to(torch.float32)
y_test_tensor = y_test_tensor.to(torch.float32)



print(x_train_tensor.dtype, y_train_tensor.dtype)

torch.float32 torch.float32


In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels


  def __len__(self):
    return len(self.features)


  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(x_train_tensor, y_train_tensor)
test_dataset = CustomDataset(x_test_tensor, y_test_tensor)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=True)

## Defining the model

In [ ]:
# print(x_train_tensor.shape)

class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out



In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
# create model
model = MySimpleNN(x_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

loss_function = nn.BCELoss()

In [ ]:

# define loop
for epoch in range(epochs):
  for batch_features, batch_labels in train_dataloader:

    # forward pass
    y_pred = model(batch_features)

    # loss calculate
    loss = loss_function(y_pred, batch_labels.unsqueeze(-1))
    optimizer.zero_grad()

    # backward pass
    loss.backward()
    optimizer.step()



  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, loss: {loss.item()}')

Epoch: 1, loss: 0.0003574533329810947
Epoch: 2, loss: 0.3619667589664459
Epoch: 3, loss: 0.0004495796747505665
Epoch: 4, loss: 0.0002216819702880457
Epoch: 5, loss: 0.01013889443129301
Epoch: 6, loss: 0.00020114939252380282
Epoch: 7, loss: 0.5200145840644836
Epoch: 8, loss: 0.00016774154209997505
Epoch: 9, loss: 0.023433512076735497
Epoch: 10, loss: 1.9550514480215497e-05
Epoch: 11, loss: 0.002800209214910865
Epoch: 12, loss: 2.9237398848636076e-05
Epoch: 13, loss: 0.0
Epoch: 14, loss: 0.0005138047854416072
Epoch: 15, loss: 1.1097336383159018e-08
Epoch: 16, loss: 8.71457887114957e-05
Epoch: 17, loss: 7.923457029379577e-12
Epoch: 18, loss: 0.0
Epoch: 19, loss: 0.0
Epoch: 20, loss: 1.0743425491455127e-06
Epoch: 21, loss: 3.0672842967760516e-06
Epoch: 22, loss: 0.0
Epoch: 23, loss: 0.029922695830464363
Epoch: 24, loss: 0.9674293398857117
Epoch: 25, loss: 4.768372718899627e-07
Epoch: 26, loss: 0.2063409388065338
Epoch: 27, loss: 0.017777355387806892
Epoch: 28, loss: 1.2981246300114435e-06


## Evaluation

In [ ]:
# Model evaluation using test_loader

In [ ]:
model.eval()    # set the model to evaluation mode
accuracy_list = []


with torch.no_grad():
  for batch_features, batch_labels in test_dataloader:

    # forward pass
    y_pred = model(batch_features)
    y_pred = (y_pred > 0.5).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean()
    accuracy_list.append(batch_accuracy)

# calculate the overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Overall accuracy: {overall_accuracy.item()}')

Overall accuracy: 0.9561403393745422
